# HELIOS Wavefront Propagation Demo

This notebook demonstrates the optical propagation capabilities of HELIOS. You can interactively explore different propagation regimes and observe how the wavefront evolves.

## Propagation Methods Overview

HELIOS implements several algorithms to simulate light propagation, each suitable for different conditions:

| Method | Type | Best For | Regime | Complexity | Output Grid |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Fraunhofer** | FFT | Far-field / Focal Plane | $z \approx f$ or $z \to \infty$ | $O(N \log N)$ | Scaled $(\lambda f / D)$ |
| **Fresnel (FFT)** | FFT | Focusing / Intermediate | Near-field / Paraxial | $O(N \log N)$ | Scaled $(\lambda z / D)$ |
| **ASM** | FFT | Free Space | Exact (Scalar) | $O(N \log N)$ | Fixed (Same as Input) |
| **S-ASM** | MFT | Zooming / Arbitrary | Exact (Scalar) | $O(N^2)$ (or optimized) | Arbitrary |
| **RS Direct** | Sum | Truth Verification | Exact (Vector/Scalar) | $O(N^4)$ | Arbitrary |

### Automatic Selection Logic (regime='Auto')

The propagation method is selected based on geometric matches and physical validity constraints ($z_{crit}$):

1.  **Fresnel Regime**: Selected if $L_{out}$ matches natural scaling AND $z > z_{min} = D^2/(N\lambda)$. (Avoids aliasing of chirp).
2.  **ASM Regime**: Selected if $L_{out} \approx D$ AND $z < z_{max} = N \Delta x^2 / \lambda$. (Avoids aliasing of transfer function).
3.  **S-ASM (Scaled ASM)**: Selected if the above conditions fail (e.g., custom zoom, or outside validity range of single-FFT methods). Robust fallback.

*Note: **Fraunhofer** is never enabled automatically.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
import ipywidgets as widgets
from IPython.display import display, clear_output
import helios
from helios import Wavefront

## Interactive Simulation

In [ ]:
# Create output widget
output_mono = widgets.Output()

# --- Controls ---

style = {'description_width': 'initial'}

wavelength_slider = widgets.FloatSlider(
    value=633, min=400, max=2000, step=10,
    description='Wavelength (nm):', style=style, continuous_update=False
)

npix_slider = widgets.Dropdown(
    options=[64, 128, 256, 512],
    value=128,
    description='Grid Size (N):', style=style
)

size_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description='Pupil Size (m):', style=style, continuous_update=False
)

focal_length_slider = widgets.FloatSlider(
    value=10.0, min=1.0, max=100.0, step=1.0,
    description='Focal Length (m):', style=style, continuous_update=False
)

use_lens_checkbox = widgets.Checkbox(
    value=True,
    description='Apply Lens',
    style=style
)

distance_slider = widgets.FloatSlider(
    value=10.0, min=0.1, max=200.0, step=0.1,
    description='Prop. Distance (m):', style=style, continuous_update=False
)

output_size_slider = widgets.FloatSlider(
    value=10.0, min=0.1, max=100.0, step=0.1,
    description='Output FOV (mm):', style=style, continuous_update=False
)

regime_selector = widgets.Dropdown(
    options=['Auto', 'Fraunhofer', 'Fresnel', 'ASM', 'SCASM', 'RS_Direct', 'Fresnel_Custom'],
    value='Auto',
    description='Method:', style=style
)

def update_simulation(λ_nm, N, pupil_size_m, f_m, use_lens, z_m, fov_mm, regime):
    with output_mono:
        clear_output(wait=True)
        
        # Units
        wavelength = λ_nm * u.nm
        size = pupil_size_m * u.m
        focal_length = f_m * u.m if use_lens else None
        distance = z_m * u.m
        output_size = fov_mm * u.mm
        
        # 1. Create Input Wavefront
        wf = Wavefront(wavelength=wavelength, size=size, npix=N)
        
        # Simple Circular Aperture with Obscuration
        y, x = wf.coordinates()
        r = np.sqrt(x**2 + y**2)
        mask = (r <= size/2) & (r >= size/6) # 33% obscuration
        wf[:] = mask.astype(complex)
        
        # 2. Propagate
        try:
            if regime.lower() == 'auto':
                regime_arg = None
            else:
                regime_arg = regime.lower()
                
            wf_out = wf.propagate(
                distance=distance,
                focal_length=focal_length,
                output_size=output_size,
                output_npix=N,
                regime=regime_arg
            )
            
            # 3. Visualization
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            
            # Input
            img_in = wf.intensity
            axes[0].imshow(img_in, cmap='gray', extent=[-pupil_size_m/2, pupil_size_m/2, -pupil_size_m/2, pupil_size_m/2])
            axes[0].set_title(f"Input Pupil (D={pupil_size_m}m)")
            axes[0].set_xlabel("x [m]")
            axes[0].set_ylabel("y [m]")
            
            # Output
            img_out = wf_out.intensity
            # Log scale for PSF
            img_out_log = np.log10(img_out + 1e-15)
            
            half_fov = fov_mm / 2
            im = axes[1].imshow(img_out_log, cmap='inferno', extent=[-half_fov, half_fov, -half_fov, half_fov])
            # Get used method from history if available or infer
            method_used = regime
            if hasattr(wf_out, 'history') and wf_out.history:
                 method_used = wf_out.history[-1]
            
            axes[1].set_title(f"Output\n{method_used}")
            axes[1].set_xlabel("x [mm]")
            plt.colorbar(im, ax=axes[1], label="Log Intensity")
            
            # Metrics
            energy_in = wf.integrated_intensity
            energy_out = wf_out.integrated_intensity
            
            print(f"--- Parameters ---")
            print(f"Focal Length: {focal_length}")
            print(f"Distance: {distance}")
            print(f"Output Scale: {wf_out.pixel_scale.to(u.mm):.2e}/pix")
            print(f"--- Metrics ---")
            print(f"Energy Input: {energy_in:.2e}")
            print(f"Energy Output: {energy_out:.2e}")
            print(f"Ratio (Out/In): {energy_out/energy_in:.4f}")
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error during propagation: {e}")

# Layout
ui = widgets.HBox([
    widgets.VBox([wavelength_slider, npix_slider, size_slider, widgets.HBox([use_lens_checkbox, focal_length_slider])]),
    widgets.VBox([distance_slider, output_size_slider, regime_selector])
])

interactive_plot = widgets.interactive_output(update_simulation, {
    'λ_nm': wavelength_slider,
    'N': npix_slider,
    'pupil_size_m': size_slider,
    'f_m': focal_length_slider,
    'use_lens': use_lens_checkbox,
    'z_m': distance_slider,
    'fov_mm': output_size_slider,
    'regime': regime_selector
})

display(ui, output_mono)
display(interactive_plot)